# 🚀 大語言模型量化分析與基準評估 (Google Colab 雲端加速版)

本筆記本提供在 **Google Colab (T4 / A100 / L4 GPU)** 上一鍵運行大語言模型（LLM）的**官方標準 PPL 困惑度評估**與**量化前特徵矩陣診斷**。

### 💡 為什麼推薦在 Google Colab 運行？
1. **遠離本地 Mac OOM 與黑屏重開機**：Colab 具備獨立 NVIDIA GPU 與 Dedicated VRAM，不再受限於 Apple Silicon 的 Metal 記憶體崩潰。
2. **完整 2048 Context Length 評估**：標準 T4 (16GB VRAM) 即可無痛容納 Qwen 3.5 2B / 4B 與 Gemma 4 的 2048 長窗口注意力運算。
3. **即時視覺化**：可在 Notebook 內直接繪製神經網路各層的激活值尖刺（Outlier Spikes）與峰均比（PAR）分佈圖。

---

## 1. 檢查 GPU 資源 (NVIDIA-SMI)

In [ ]:
!nvidia-smi

## 2. 掛載 Google Drive (可選，用於持久化保存權重與結果)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 建立結果保存目錄
!mkdir -p /content/drive/MyDrive/model_quantization_results

## 3. 環境與依賴安裝

In [ ]:
# 若從 GitHub Clone 倉庫，請替換為你的 Repository URL
# !git clone https://github.com/your-username/model_quantization.git
# %cd model_quantization

# 安裝核心依賴
!pip install -q "transformers>=4.40.0" "accelerate>=0.28.0" "datasets>=2.18.0" sentencepiece protobuf scipy matplotlib tqdm

## 4. 設定 Hugging Face 存取 Token (存取 Gemma / Llama Gated 模型必備)
如果是評估 Qwen 模型可直接使用；若是 Gemma 4 或 Llama 3.2，請至 Hugging Face 取得 Access Token。

In [ ]:
from huggingface_hub import login
# 填入你的 Hugging Face Token，若無需授權模型可跳過此單元格
# login('hf_xxxxxxxxxxxxxxxxxxxx')

## 5. 純文字高效推論驗證 (Demo Inference)
驗證純文字剝離載入與 CUDA 推論速度。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3.5-2B"  # 可自由更換為 Qwen/Qwen3.5-4B 或 google/gemma-4-E2B-it
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"正在載入模型 {model_id} 至 {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

prompt = "請用繁體中文以三點總結模型量化（Quantization）在大語言模型部署中的重要優勢："
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)

print("\n--- 生成結果 ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 6. 官方標準 WikiText-2 PPL (困惑度) 基準評估
嚴格遵循 Hugging Face 官方標準 Strided Sliding Window 演算法（2048 Context Length, 512 Stride, -100 Context Masking）。

In [ ]:
import math
import time
import torch
from datasets import load_dataset
from tqdm import tqdm

def run_colab_ppl_benchmark(
    model,
    tokenizer,
    device="cuda",
    max_length=2048,
    stride=512,
    max_steps=None,
    dataset_name="wikitext",
    dataset_config="wikitext-2-raw-v1",
    split="test"
):
    dataset = load_dataset(dataset_name, dataset_config, split=split)
    full_text = "\n\n".join([t for t in dataset["text"] if t.strip()])
    encodings = tokenizer(full_text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    print(f"評估資料集 Token 總數: {seq_len:,}")
    print(f"窗口長度 (max_len): {max_length}, 滑動步長 (stride): {stride}")

    nlls = []
    prev_end_loc = 0
    step_count = 0
    start_time = time.time()

    total_possible = (seq_len + stride - 1) // stride
    total_steps = min(total_possible, max_steps) if max_steps else total_possible
    pbar = tqdm(total=total_steps, desc="WikiText-2 PPL 評估")

    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids=input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss.item() * trg_len
            nlls.append(neg_log_likelihood)

        prev_end_loc = end_loc
        step_count += 1
        pbar.update(1)

        del outputs, input_ids, target_ids
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if end_loc == seq_len or (max_steps and step_count >= max_steps):
            break

    pbar.close()
    total_evaluated_tokens = prev_end_loc
    total_nll = sum(nlls)
    ppl = math.exp(total_nll / total_evaluated_tokens) if total_evaluated_tokens > 0 else float('nan')
    elapsed = time.time() - start_time

    print("\n" + "=" * 60)
    print(f"🏆 WikiText-2 PPL (困惑度) : {ppl:.4f}")
    print(f"⚡ 評估耗時              : {elapsed:.2f} 秒")
    print(f"🚀 處理吞吐量            : {total_evaluated_tokens / elapsed:.2f} tokens/s")
    print(f"📝 總評估 Token 數       : {total_evaluated_tokens:,}")
    print("=" * 60)
    return ppl, elapsed

# 執行完整評估 (在 Colab T4 16GB 上可直接跑 2048 窗口，亦可設置 max_steps=40 加速)
ppl_score, elapsed_time = run_colab_ppl_benchmark(
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_length=2048,
    stride=512,
    max_steps=40  # 設為 None 即可跑全量資料集
)

## 7. 激活值離群點矩陣體檢 (Activation Outlier Profiling & Visualization)
利用 Forward Hook 捕獲每一層神經網路的特徵張量，分析通道尖刺與峰均比（PAR），並可視化繪圖。

In [ ]:
import matplotlib.pyplot as plt

def find_decoder_layers(model):
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.ModuleList) and len(module) > 5:
            if "language_model" in name or ("layers" in name and "vision" not in name and "audio" not in name):
                return list(module)
    raise RuntimeError("無法定位 Transformer Decoder Layers")

layers = find_decoder_layers(model)
num_layers = len(layers)
records = {}
hooks = []

def make_hook(idx):
    def hook_fn(module, args, output):
        feat = output[0] if isinstance(output, tuple) else output
        feat = feat.detach().float().abs()
        g_max = feat.max().item()
        g_mean = feat.mean().item()
        par = g_max / (g_mean + 1e-8)
        records[idx] = {"layer": idx, "max": g_max, "mean": g_mean, "par": par}
    return hook_fn

for i, layer in enumerate(layers):
    hooks.append(layer.register_forward_hook(make_hook(i)))

# 執行校準前向傳播
sample_input = tokenizer("Large language models exhibit significant activation outliers in deeper layers.", return_tensors="pt").to(device)
with torch.no_grad():
    model(**sample_input)

for h in hooks:
    h.remove()

# 繪製各層峰均比 (PAR) 圖表
layer_indices = [records[i]["layer"] for i in range(num_layers)]
pars = [records[i]["par"] for i in range(num_layers)]
max_vals = [records[i]["max"] for i in range(num_layers)]

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(layer_indices, pars, marker='o', color='#e74c3c', linewidth=2)
plt.axhline(y=15, color='gray', linestyle='--', label='Outlier Threshold (15x)')
plt.title(f"{model_id} 各層峰均比 (Peak-to-Average Ratio)")
plt.xlabel("Layer Index")
plt.ylabel("PAR (Max / Mean)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.bar(layer_indices, max_vals, color='#3498db', alpha=0.8)
plt.title(f"{model_id} 各層激活值最大峰值 (Global Max Activation)")
plt.xlabel("Layer Index")
plt.ylabel("Max Activation Value")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 儲存實驗數據 (CSV 匯出)
將測試結果自動存回 Google Drive，確保離線後資料依然保留。

In [ ]:
import pandas as pd

df_results = pd.DataFrame([{
    "model_id": model_id,
    "ppl_wikitext2": ppl_score,
    "elapsed_sec": elapsed_time,
    "max_par": max(pars),
    "max_activation_value": max(max_vals),
}])

csv_path = "/content/drive/MyDrive/model_quantization_results/benchmark_summary.csv"
df_results.to_csv(csv_path, mode="a", header=not pd.io.common.file_exists(csv_path), index=False)
print(f"✅ 實驗紀錄已成功保存至 Google Drive: {csv_path}")
display(df_results)